# OncoEvidence — Part 1 of 4: Foundations
## Knowledge graph + candidate ranking (Aim 1)

> **4-part learning track:** **Part 1 — Foundations** → **Part 2 — Mechanisms** → **Part 3 — Evidence & the graph's edge** → **Part 4 — Full self-contained notebook**.
>
> Each part is standalone and Colab-ready (Runtime → Change runtime type → **GPU**). The toolbox cells (`L1`, `L2`, …) are the *same* building blocks across parts and match the full Part 4 notebook — so as you progress you are assembling the complete system, organically, one capability at a time.
>
> ⚠️ Hypothesis-generating research, **not medical advice.**

### What you'll learn in Part 1
- **What PrimeKG is**: a large *heterogeneous* biomedical knowledge graph (~129k nodes, ~8.1M edges across drugs, diseases, genes/proteins, pathways, …) and how it's structured.
- **How to make it model-ready**: derive node features from names, and — crucially — build **leakage-safe** train/val/test splits (transductive, cold-disease, cold-drug).
- **How candidate generation works**: train a heterogeneous **Graph Neural Network** to rank (drug, disease) pairs, and benchmark it honestly against a **tuned XGBoost**, an **MLP**, and a **knowledge-graph embedding (KGE)**.
- **The first honest finding**: on pure link ranking, a tuned XGBoost on the same features **ties or beats** the GNN. That's the whole motivation for Parts 2–4: if ranking alone doesn't justify the graph, what does? (Answer: *mechanism*.)

## 0. Configuration switches

This single cell controls how heavy the run is. Two big knobs:

- **`FAST_MODE`** — `True` runs a quick demo (1 random seed, fewer training epochs,
  cheap "hashing" text features) so the whole notebook finishes in a few minutes.
  `True` is plenty to *see every result qualitatively*. Set it to `False` to reproduce
  the **published numbers** (5 seeds, real SentenceTransformer embeddings, XGBoost
  hyper-parameter tuning) — that takes a few hours on a GPU.
- **`RUN_LLM`** — `True` enables the LLM reviewer/judge. You must paste an
  OpenAI-compatible API key. With it off, the literature verifier still works using a
  transparent lexical (keyword-matching) fallback, so the notebook runs key-free.

Everything else below are derived budgets (epochs, seed counts, sample sizes) that
scale automatically with `FAST_MODE`.

In [ ]:
# ── Master switches ──────────────────────────────────────────────────────────
FAST_MODE = True   # True: quick demo (minutes). False: full published reproduction (hours).
RUN_LLM   = False  # True: call a real LLM reviewer (needs an API key below).

import os

# ── LLM credentials (only used when RUN_LLM=True) ────────────────────────────
# The LLM client is OpenAI-compatible. The default base URL points at OpenRouter,
# which proxies many models; for the official OpenAI API use https://api.openai.com/v1.
os.environ.setdefault("ONCO_LLM_API_KEY", "")               # <-- paste your key here to enable the LLM
ONCO_LLM_BASE_URL = "https://openrouter.ai/api/v1"          # OpenAI: https://api.openai.com/v1
ONCO_LLM_MODEL    = "openai/gpt-4o-mini"                    # a small, cheap, capable judge model
if RUN_LLM and os.environ["ONCO_LLM_API_KEY"]:
    os.environ["ONCO_LLM_BASE_URL"] = ONCO_LLM_BASE_URL
    os.environ["ONCO_LLM_MODEL"]    = ONCO_LLM_MODEL

# ── Training / evaluation budgets (auto-scale with FAST_MODE) ────────────────
# In fast mode we use a single seed and short training; in full mode we average over
# 5 seeds and train longer, which is what the paper's numbers were produced with.
SEEDS              = [0] if FAST_MODE else [0, 1, 2, 42, 123]  # random seeds we average over
ABLATION_SEEDS     = [0] if FAST_MODE else [0, 1, 2]           # seeds for the (expensive) ablations
GNN_EPOCHS         = 12 if FAST_MODE else 50                   # GNN training epochs (early-stopped)
MLP_EPOCHS         = 50 if FAST_MODE else 200                  # MLP baseline epochs
KGE_EPOCHS         = 80 if FAST_MODE else 300                  # knowledge-graph-embedding epochs
PATIENCE           = 6 if FAST_MODE else 10                    # early-stopping patience (val AUROC)
XGB_TUNE           = (not FAST_MODE)                           # tune XGBoost only in full mode
XGB_TRIALS         = 5 if FAST_MODE else 8                     # Optuna trials when tuning
XGB_ESTIMATORS     = 200 if FAST_MODE else 400                 # XGBoost trees (upper bound; early-stopped)
USE_FALLBACK_FEATS = FAST_MODE                                 # hashing feats (instant) vs ST embeddings (slower)
N_SEP              = 120 if FAST_MODE else 400                 # pairs per group in the separation test
REC_EPOCHS         = 12 if FAST_MODE else 60                   # mechanism-recovery training epochs
REC_SEEDS          = [0] if FAST_MODE else [0, 1, 2]           # seeds for mechanism recovery

print(f"FAST_MODE={FAST_MODE}  RUN_LLM={RUN_LLM}")
print(f"seeds={SEEDS}  gnn_epochs={GNN_EPOCHS}  xgb_tune={XGB_TUNE}  fallback_features={USE_FALLBACK_FEATS}")

## 1. Install dependencies

On Google Colab, PyTorch and the usual scientific stack are already installed, so we
only add the project-specific extras:

- **`torch-geometric`** — the Graph Neural Network library (heterogeneous graph layers).
- **`xgboost`** — gradient-boosted trees, our strongest tabular baseline.
- **`optuna`** — hyper-parameter search for XGBoost (full mode only).
- **`sentence-transformers` / `transformers`** — turn node *names* (e.g. "imatinib")
  into numeric embedding vectors used as node features.
- **`scikit-learn`, `pandas`, `numpy`, `scipy`** — metrics, tables, stats.
- **`matplotlib`, `seaborn`** — the in-notebook plots.
- **`requests`** — call the Europe PMC / DrugMechDB / mygene web APIs.
- **`pyyaml`, `tqdm`** — parse DrugMechDB YAML; progress bars.

The `-q` flag keeps the install quiet.

In [ ]:
# Install only the extras Colab doesn't ship with. (Safe to re-run; pip is idempotent.)
%pip install -q torch-geometric xgboost optuna "sentence-transformers>=2.2.0" "transformers>=4.40,<5" scikit-learn pandas numpy scipy matplotlib seaborn requests tqdm pyyaml
print("Dependencies installed.")

In [ ]:
# Pick the compute device. A GPU makes GNN training ~10-50x faster than CPU.
import torch
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    # Not fatal — everything still runs on CPU, just slowly. Prefer enabling a GPU runtime.
    print("WARNING: no GPU detected. Use Runtime > Change runtime type > GPU for reasonable speed.")

## Toolbox — run these once

The cells below (`L…`) only *define* the functions this part needs; running them is fast and prints little. They carry the same `L1…L12` labels as the full notebook so the pieces line up across the track. Read them if you're curious how things work — otherwise just run them top to bottom and head to the workflow below.

There is no `import` from the project source anywhere: every part is fully self-contained.

### L1 · Paths, schema constants, and oncology keywords

Sets up:
- **Folders** (`data/`, `models/`, `results/`, `figures/`) where we cache the graph,
  features, and outputs so re-runs are fast.
- **PrimeKG download URL** and cache filenames.
- **Schema constants** — PrimeKG is a *heterogeneous* graph (many node/edge types).
  We name the ones we care about: `drug`, `disease`, `gene_protein`, and the three
  "therapeutic" drug↔disease relations (`indication`, `contraindication`,
  `off-label use`). These therapeutic edges are the *labels* we predict, so they must
  be handled carefully to avoid leakage.
- **`ONCOLOGY_KEYWORDS`** — a simple word list used to flag which diseases are cancers
  (so we can restrict evaluation to oncology).

In [ ]:
from pathlib import Path

# Local cache/output directories. On Colab these live under /content and reset per session.
DATA_DIR    = Path("data")     # raw PrimeKG csv + built graph + small json caches
MODELS_DIR  = Path("models")   # feature caches + LLM response cache
RESULTS_DIR = Path("results")  # metric JSONs
FIGURES_DIR = Path("figures")  # saved plots
for _d in (DATA_DIR, MODELS_DIR, RESULTS_DIR, FIGURES_DIR):
    _d.mkdir(parents=True, exist_ok=True)

# PrimeKG: a large public biomedical knowledge graph (~8M edges) hosted on Harvard Dataverse.
PRIMEKG_KG_CSV = DATA_DIR / "kg.csv"                                    # raw edge list (~980 MB)
PRIMEKG_KG_URL = "https://dataverse.harvard.edu/api/access/datafile/6180620"
HETERODATA_PT  = DATA_DIR / "primekg_hetero.pt"                         # parsed PyG graph (cached)
FEATURE_CACHE  = MODELS_DIR / "primekg_text_features.pt"               # node embeddings (cached)
LLM_CACHE_DIR  = MODELS_DIR / "llm_cache"                              # cache LLM calls (save $ + time)
UNIPROT_CACHE  = DATA_DIR / "uniprot2symbol.json"                     # UniProt->gene-symbol map (cached)

# The drug<->disease "therapeutic" relations. `indication` (drug treats disease) is our
# main prediction target; all three are stripped from the message-passing graph to
# prevent the model from "seeing the answer" (label leakage).
INDICATION_REL = "indication"; CONTRAINDICATION_REL = "contraindication"; OFFLABEL_REL = "off-label use"
THERAPEUTIC_RELS = (INDICATION_REL, CONTRAINDICATION_REL, OFFLABEL_REL)

# Node-type names exactly as they appear in PrimeKG after normalization.
DRUG_TYPE = "drug"; DISEASE_TYPE = "disease"; GENE_TYPE = "gene_protein"

# SentenceTransformer model used to embed node names into vectors (full mode).
TEXT_MODEL = "all-MiniLM-L6-v2"

# Substring rules to flag a disease as cancer. Crude but effective on PrimeKG names.
ONCOLOGY_KEYWORDS = (
    "cancer","carcinoma","sarcoma","neoplasm","tumor","tumour","leukemia","leukaemia",
    "lymphoma","melanoma","glioma","glioblastoma","myeloma","blastoma","adenoma",
    "malignant","metastat","oncocytoma","mesothelioma","astrocytoma","teratoma","carcinosarcoma",
)
print("L1 constants ready")

### L2 · Download PrimeKG and build the heterogeneous graph

PrimeKG ships as a flat CSV edge list: each row is
`(x_type, x_id, x_name, relation, y_type, y_id, y_name)`. We convert it into a PyTorch
Geometric **`HeteroData`** object — the in-memory representation a GNN consumes.

The build does three things:
1. **Register nodes.** Assign every unique `(type, id)` a contiguous integer index per
   type, and remember its human-readable name. (GNNs operate on integer indices.)
2. **Bucket edges by type.** Group rows into `(source_type, relation, dest_type)`
   "edge types", each stored as a `[2, num_edges]` index tensor.
3. **Flag oncology diseases** using the keyword list, stored as a boolean mask.

We cache the result to `primekg_hetero.pt` so this minutes-long parse only happens once.

In [ ]:
import sys, re, urllib.request
from collections import defaultdict
import pandas as pd
import torch
from torch_geometric.data import HeteroData

def download_primekg(dest=PRIMEKG_KG_CSV, force=False):
    '''Download the PrimeKG edge list if we don't already have it (~980 MB, one-time).'''
    dest = Path(dest)
    if dest.exists() and not force and dest.stat().st_size > 0:
        print(f"PrimeKG already present: {dest} ({dest.stat().st_size/1e6:.0f} MB)"); return dest
    dest.parent.mkdir(parents=True, exist_ok=True)
    print(f"Downloading PrimeKG -> {dest} (~980 MB, this can take a few minutes)")
    def _progress(block_num, block_size, total_size):  # simple percent printout
        if total_size > 0:
            sys.stdout.write(f"\r  {min(100.0, block_num*block_size*100.0/total_size):5.1f}%"); sys.stdout.flush()
    urllib.request.urlretrieve(PRIMEKG_KG_URL, dest, _progress); sys.stdout.write("\n")
    print(f"Saved {dest} ({dest.stat().st_size/1e6:.0f} MB)"); return dest

# Normalize free-text type/relation strings into clean snake_case tokens (e.g. "Drug Protein"
# -> "drug_protein"), so the graph schema is consistent regardless of source capitalization.
def _norm_type(t): return re.sub(r"[^0-9a-zA-Z]+", "_", str(t).strip().lower()).strip("_")
def _norm_rel(r):  return re.sub(r"[^0-9a-zA-Z]+", "_", str(r).strip().lower()).strip("_")
def _is_oncology(name): return any(k in str(name).lower() for k in ONCOLOGY_KEYWORDS)

def build_hetero_from_primekg(kg_csv=PRIMEKG_KG_CSV, output_path=HETERODATA_PT, save=True):
    '''Parse the PrimeKG CSV into a PyG HeteroData graph and (optionally) cache it.'''
    kg_csv = Path(kg_csv)
    if not kg_csv.exists():
        raise FileNotFoundError(f"{kg_csv} not found; run download_primekg() first.")
    print(f"Loading {kg_csv} ...")
    df = pd.read_csv(kg_csv, dtype=str, low_memory=False)   # read everything as strings; ids are mixed
    print(f"  {len(df):,} raw edges")
    # Normalize type/relation columns up front (vectorized, fast).
    df["x_type_n"] = df["x_type"].map(_norm_type); df["y_type_n"] = df["y_type"].map(_norm_type)
    df["rel_n"] = df["relation"].map(_norm_rel)

    # --- Step 1: assign each (type, original_id) a contiguous per-type integer index ---
    node_id_to_idx = defaultdict(dict)    # type -> {original_id: int_index}
    node_names = defaultdict(list)        # type -> [name per index]
    node_ids = defaultdict(list)          # type -> [original_id per index]
    def _register(t, i, nm):
        m = node_id_to_idx[t]
        if i not in m:                    # first time we see this node id
            m[i] = len(m)                 # next free index for this type
            node_names[t].append("" if nm is None else str(nm))
            node_ids[t].append(str(i))
        return m[i]
    # A node can appear on either side of an edge, so scan both the x- and y-columns.
    for side in ("x", "y"):
        sub = df[[f"{side}_type_n", f"{side}_id", f"{side}_name"]].drop_duplicates()
        for t, i, nm in sub.itertuples(index=False):
            _register(t, i, nm)

    data = HeteroData()
    # Attach per-type metadata (count, original ids, names) to the graph.
    for nt, m in node_id_to_idx.items():
        data[nt].num_nodes = len(m); data[nt].node_ids = node_ids[nt]; data[nt].node_names = node_names[nt]

    # --- Step 2: group edges into (src_type, relation, dst_type) buckets ---
    buckets = defaultdict(list)
    for x_t, x_id, y_t, y_id, rel in df[["x_type_n","x_id","y_type_n","y_id","rel_n"]].itertuples(index=False):
        buckets[(x_t, rel, y_t)].append((node_id_to_idx[x_t][x_id], node_id_to_idx[y_t][y_id]))
    for et, pairs in buckets.items():
        # edge_index has shape [2, num_edges]: row 0 = source indices, row 1 = dest indices.
        data[et].edge_index = torch.tensor(pairs, dtype=torch.long).t().contiguous()

    # --- Step 3: mark which diseases are cancers (used to restrict oncology evaluation) ---
    if DISEASE_TYPE in data.node_types:
        names = data[DISEASE_TYPE].node_names
        data[DISEASE_TYPE].is_oncology = torch.tensor([_is_oncology(n) for n in names], dtype=torch.bool)
        print(f"  oncology diseases flagged: {int(data[DISEASE_TYPE].is_oncology.sum())} / {len(names)}")

    total_nodes = sum(int(data[t].num_nodes) for t in data.node_types)
    total_edges = sum(int(data[e].edge_index.size(1)) for e in data.edge_types)
    print(f"  {len(data.node_types)} node types ({total_nodes:,} nodes), "
          f"{len(data.edge_types)} edge types ({total_edges:,} edges)")
    if save:
        torch.save(data, Path(output_path)); print(f"Saved parsed graph -> {output_path}")
    return data
print("L2 ready")

### L3 · Node features, graph loader, and summary

A GNN needs a **feature vector** for every node. We derive features purely from each
node's **name** (e.g. the drug "imatinib", the gene "BCR"):

- **Preferred (full mode):** a *SentenceTransformer* turns each name into a dense
  semantic embedding, so similar names land near each other in vector space.
- **Fallback (fast mode / offline):** a deterministic *hashing* embedding of the name's
  character n-grams. Instant, no model download, no GPU — good enough for a demo.

This cell also provides:
- `load_primekg(...)` — build-or-load the cached graph, attach features, and locate the
  `indication`/`contraindication` target edge types.
- `graph_summary(...)` — a human-readable one-glance description of the loaded graph.

The features are cached to disk so we never recompute them.

In [ ]:
import hashlib
import numpy as np

DEFAULT_HASH_DIM = 256   # dimensionality of the fallback hashing features

def _node_texts(data, nt):
    '''Build a short descriptive string per node, e.g. 'drug: imatinib'.'''
    store = data[nt]; names = getattr(store, "node_names", None); n = int(store.num_nodes)
    pretty = nt.replace("_", " ")
    if names is None:
        return [f"{pretty} {i}" for i in range(n)]
    out = []
    for i in range(n):
        raw = "" if (i >= len(names) or names[i] is None) else str(names[i]).strip()
        out.append(f"{pretty}: {raw}" if raw else f"{pretty} {i}")
    return out

def _hash_embed(texts, dim=DEFAULT_HASH_DIM):
    '''Deterministic 'feature hashing' embedding: map char n-grams into a fixed vector.

    This needs no model and no network. Each token and 3-gram is hashed to a bucket and
    adds +/-1; the vector is L2-normalized. Names sharing substrings get similar vectors.
    '''
    vecs = np.zeros((len(texts), dim), dtype=np.float32)
    for row, text in enumerate(texts):
        toks = re.findall(r"[a-z0-9]+", text.lower()); grams = list(toks)
        for tok in toks:                                   # add character 3-grams of each token
            padded = f"#{tok}#"; grams.extend(padded[i:i+3] for i in range(len(padded)-2))
        for g in grams:
            h = int(hashlib.md5(g.encode()).hexdigest(), 16)
            vecs[row, h % dim] += 1.0 if (h // dim) % 2 == 0 else -1.0   # signed hashing reduces collisions
    norms = np.linalg.norm(vecs, axis=1, keepdims=True); norms[norms == 0] = 1.0
    return vecs / norms

def _st_embed(texts_by_type, model_name):
    '''Embed node names with a SentenceTransformer; return None to trigger the hashing fallback.'''
    try:
        from sentence_transformers import SentenceTransformer
    except Exception as exc:
        print(f"  [features] sentence-transformers unavailable ({exc}); using hashing fallback"); return None
    try:
        model = SentenceTransformer(model_name); out = {}
        for nt, texts in texts_by_type.items():
            out[nt] = model.encode(texts, batch_size=512, normalize_embeddings=True,
                                   show_progress_bar=False, convert_to_numpy=True).astype(np.float32)
        return out
    except Exception as exc:
        print(f"  [features] SentenceTransformer failed ({exc}); using hashing fallback"); return None

def build_text_features(data, cache_path=FEATURE_CACHE, model_name=TEXT_MODEL, force_fallback=False):
    '''Attach an `.x` feature matrix to every node type, using cache when available.'''
    cache_path = Path(cache_path) if cache_path is not None else None
    # Keep hashing-features in a separate cache file so the two modes never collide.
    if cache_path is not None and force_fallback:
        cache_path = cache_path.with_name(cache_path.stem + "_hash" + cache_path.suffix)
    # Reuse a cache only if its row counts match the current graph exactly.
    if cache_path is not None and cache_path.exists():
        cached = torch.load(cache_path, weights_only=False)
        if all(nt in cached and cached[nt].shape[0] == int(data[nt].num_nodes) for nt in data.node_types):
            for nt in data.node_types: data[nt].x = cached[nt].float()
            print(f"  [features] loaded from cache (dim={next(iter(cached.values())).shape[1]})"); return data
    texts = {nt: _node_texts(data, nt) for nt in data.node_types}
    emb = None if force_fallback else _st_embed(texts, model_name)
    if emb is None:                                         # offline / fast path
        emb = {nt: _hash_embed(t) for nt, t in texts.items()}; src = f"hashing (dim={DEFAULT_HASH_DIM})"
    else:
        src = f"{model_name} (dim={next(iter(emb.values())).shape[1]})"
    tensors = {}
    for nt in data.node_types:
        x = torch.from_numpy(np.ascontiguousarray(emb[nt])).float(); data[nt].x = x; tensors[nt] = x
    print(f"  [features] built via {src}")
    if cache_path is not None:
        cache_path.parent.mkdir(parents=True, exist_ok=True); torch.save(tensors, cache_path)
    return data

def _find_target_edge(data, relation):
    '''Find the (drug, relation, disease) edge type for a therapeutic relation, drug-first.'''
    rel_n = _norm_rel(relation); cands = []
    for et in data.edge_types:
        s, r, d = et
        if {s, d} == {DRUG_TYPE, DISEASE_TYPE} and r == rel_n: cands.append(et)
    if not cands: return None
    for et in cands:
        if et[0] == DRUG_TYPE: return et    # prefer the drug->disease orientation
    return cands[0]

def load_primekg(pt_path=HETERODATA_PT, with_features=True, force_fallback_features=False, build_if_missing=True):
    '''Top-level loader: build-or-load the graph, attach features, return target edge types.'''
    pt_path = Path(pt_path)
    if not pt_path.exists():
        if not build_if_missing: raise FileNotFoundError(f"{pt_path} not found.")
        data = build_hetero_from_primekg(save=True)
    else:
        import warnings
        with warnings.catch_warnings():
            warnings.simplefilter("ignore"); data = torch.load(pt_path, weights_only=False)
    # Some cached graphs may miss num_nodes on isolated types; backfill defensively.
    for nt in data.node_types:
        if not hasattr(data[nt], "num_nodes") or data[nt].num_nodes is None:
            names = getattr(data[nt], "node_names", None)
            data[nt].num_nodes = len(names) if names is not None else 0
    if with_features:
        build_text_features(data, force_fallback=force_fallback_features)
    targets = {"indication": _find_target_edge(data, INDICATION_REL),
               "contraindication": _find_target_edge(data, CONTRAINDICATION_REL)}
    return data, targets

def graph_summary(data, targets):
    '''Return a multi-line human-readable summary of node types, edges, and targets.'''
    lines = ["PrimeKG graph:"]; total_nodes = total_edges = 0
    for nt in data.node_types:
        n = int(data[nt].num_nodes); total_nodes += n
        dim = int(data[nt].x.size(1)) if "x" in data[nt] else None
        extra = f" feat_dim={dim}" if dim else ""
        if nt == DISEASE_TYPE and "is_oncology" in data[nt]:
            extra += f" oncology={int(data[nt].is_oncology.sum())}"
        lines.append(f"  node {nt}: {n}{extra}")
    for et in data.edge_types: total_edges += int(data[et].edge_index.size(1))
    lines.append(f"  totals: {total_nodes:,} nodes, {total_edges:,} edges, {len(data.edge_types)} edge types")
    for name, et in targets.items():
        lines.append(f"  target [{name}]: {et} = {int(data[et].edge_index.size(1))} edges" if et
                     else f"  target [{name}]: NOT FOUND")
    return "\n".join(lines)
print("L3 ready")

### L4 · The models

We define four link-prediction models that all answer "how likely is this (drug,
disease) edge?":

- **`HeteroGNN`** — the star. It runs **message passing** over the heterogeneous graph:
  each node repeatedly aggregates information from its neighbors of every relation type
  (via `SAGEConv` wrapped in `HeteroConv`), producing a context-aware embedding per node.
  An `EdgeMLPDecoder` then scores a (drug, disease) pair from their two embeddings.
  It also has a `MechanismHead` used later for the mechanism-recovery task — it scores a
  (drug, **gene**, disease) triple, i.e. "is this gene the mechanism bridge?".
- **`FeatureMLP`** — an ablation that *ignores graph edges*: it just transforms each
  node's raw features with an MLP and scores pairs. Isolates "features alone".
- **`DistMultKGE`** — a classic knowledge-graph embedding: learns one vector per drug and
  per disease plus a relation vector, scoring a pair by a bilinear product. No features,
  no message passing — pure structural memorization.

Comparing these tells us *where the predictive signal actually comes from*.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import HeteroConv, SAGEConv

class EdgeMLPDecoder(nn.Module):
    '''Score an edge from the concatenation of its endpoints' embeddings -> one logit.'''
    def __init__(self, hidden):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(2*hidden, hidden), nn.ReLU(), nn.Linear(hidden, 1))
    def forward(self, z_src, z_dst):
        return self.net(torch.cat([z_src, z_dst], dim=-1)).squeeze(-1)

class MechanismHead(nn.Module):
    '''Score a (drug, gene, disease) triple -> one logit ('is this gene the bridge?').'''
    def __init__(self, hidden):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(3*hidden, hidden), nn.ReLU(), nn.Linear(hidden, 1))
    def forward(self, z_drug, z_gene, z_dis):
        return self.net(torch.cat([z_drug, z_gene, z_dis], dim=-1)).squeeze(-1)

class HeteroGNN(nn.Module):
    '''Heterogeneous GraphSAGE encoder + edge decoder + mechanism head.'''
    def __init__(self, node_types, edge_types, in_dims, hidden=128, num_layers=2, dropout=0.3):
        super().__init__()
        self.node_types = node_types; self.edge_types = edge_types; self.dropout = dropout
        # Project each node type's (possibly different-dim) features into a shared hidden space.
        self.proj = nn.ModuleDict({nt: nn.Linear(in_dims[nt], hidden) for nt in node_types})
        # Each layer applies one SAGEConv *per relation* and sums the messages per node.
        self.convs = nn.ModuleList([
            HeteroConv({et: SAGEConv(hidden, hidden) for et in edge_types}, aggr="sum")
            for _ in range(num_layers)
        ])
        self.decoder = EdgeMLPDecoder(hidden); self.mech_head = MechanismHead(hidden)
    def encode(self, data):
        '''Run message passing and return a dict {node_type: embeddings}.'''
        device = next(self.parameters()).device
        x_dict = {nt: self.proj[nt](data[nt].x.to(device)) for nt in self.node_types if nt in data.node_types}
        eid = {et: data[et].edge_index.to(device) for et in self.edge_types
               if et in data.edge_types and data[et].edge_index.numel() > 0}
        for conv in self.convs:
            out = conv(x_dict, eid)
            for nt in x_dict:                      # node types that received no messages: keep their value
                if nt not in out: out[nt] = x_dict[nt]
            # nonlinearity + dropout between layers
            x_dict = {nt: F.dropout(F.relu(v), p=self.dropout, training=self.training) for nt, v in out.items()}
        return x_dict
    def decode(self, z, et, edge_label_index):
        '''Score a batch of candidate edges of type `et`.'''
        s_t, _, d_t = et; dev = z[s_t].device; eli = edge_label_index.to(dev)
        return self.decoder(z[s_t][eli[0]], z[d_t][eli[1]])
    def score_mechanism(self, z, drug_i, gene_i, dis_i, dt="drug", gt="gene_protein", ct="disease"):
        '''Score (drug, gene, disease) triples for the mechanism-recovery task.'''
        dev = z[dt].device
        return self.mech_head(z[dt][drug_i.to(dev)], z[gt][gene_i.to(dev)], z[ct][dis_i.to(dev)])

class FeatureMLP(nn.Module):
    '''Edges-ignored baseline: MLP on raw node features only (no message passing).'''
    def __init__(self, node_types, in_dims, hidden=128, dropout=0.3):
        super().__init__(); self.node_types = node_types; self.dropout = dropout
        self.proj = nn.ModuleDict({nt: nn.Linear(in_dims[nt], hidden) for nt in node_types})
        self.decoder = EdgeMLPDecoder(hidden)
    def encode(self, data):
        device = next(self.parameters()).device
        return {nt: F.relu(self.proj[nt](data[nt].x.to(device))) for nt in self.node_types if nt in data.node_types}
    def decode(self, z, et, edge_label_index):
        s_t, _, d_t = et; dev = z[s_t].device; eli = edge_label_index.to(dev)
        return self.decoder(z[s_t][eli[0]], z[d_t][eli[1]])

class DistMultKGE(nn.Module):
    '''DistMult knowledge-graph embedding: learn drug/disease vectors + a relation vector.'''
    def __init__(self, src_type, dst_type, num_src, num_dst, dim=128):
        super().__init__(); self.src_type, self.dst_type = src_type, dst_type
        self.src_emb = nn.Embedding(num_src, dim); self.dst_emb = nn.Embedding(num_dst, dim)
        self.rel = nn.Parameter(torch.randn(dim) * 0.1)        # single learned relation vector
        nn.init.xavier_uniform_(self.src_emb.weight); nn.init.xavier_uniform_(self.dst_emb.weight)
    def score(self, edge_label_index):
        dev = self.rel.device; eli = edge_label_index.to(dev)
        # DistMult score = <src, rel, dst> elementwise then summed.
        return (self.src_emb(eli[0]) * self.rel * self.dst_emb(eli[1])).sum(-1)
print("L4 ready")

### L5 · Evaluation metrics

Standard link-prediction metrics, all computed from predicted scores vs binary labels:

- **AUROC** — probability a random positive scores above a random negative (0.5 = chance).
- **AUPRC** — area under precision/recall; better than AUROC when positives are rare.
- **Hits@k / MRR** — ranking quality (is the true edge near the top?).
- **Precision/Recall/F1** at the threshold that maximizes F1.

`compute_all_metrics` returns all of them in one dict and guards degenerate cases
(all-positive or all-negative label sets) by returning zeros.

In [ ]:
from sklearn.metrics import (average_precision_score, f1_score, precision_recall_curve,
                             precision_score, recall_score, roc_auc_score)

def _to_np(x):
    if isinstance(x, torch.Tensor): return x.detach().cpu().numpy()
    if isinstance(x, list): return np.array(x)
    return x

def hits_at_k(y, s, k=10):
    '''Fraction of true positives captured in the top-k highest-scoring items.'''
    y, s = _to_np(y), _to_np(s); order = np.argsort(-s)[:k]; total_pos = float(np.sum(y))
    return 0.0 if total_pos == 0 else float(np.sum(y[order]) / min(k, total_pos))

def mean_reciprocal_rank(y, s):
    '''Average of 1/rank over the positive items (higher = positives ranked nearer the top).'''
    y, s = _to_np(y), _to_np(s); ranked = y[np.argsort(-s)]; positions = np.where(ranked == 1)[0] + 1
    return 0.0 if positions.size == 0 else float(np.mean(1.0 / positions))

def optimal_threshold_metrics(y, s):
    '''Precision/recall/F1 at the score threshold that maximizes F1.'''
    prec, rec, thr = precision_recall_curve(y, s)
    f1 = 2*(prec[:-1]*rec[:-1]) / (prec[:-1]+rec[:-1]+1e-10)
    if f1.size == 0: return {"precision":0.0,"recall":0.0,"f1":0.0,"threshold":0.5}
    i = int(np.argmax(f1)); t = float(thr[i]); pred = (s >= t).astype(int)
    return {"precision":float(precision_score(y,pred,zero_division=0)),
            "recall":float(recall_score(y,pred,zero_division=0)),
            "f1":float(f1_score(y,pred,zero_division=0)),"threshold":t}

def compute_all_metrics(y, s, prefix=""):
    '''One-stop metric dict. Returns zeros if labels are degenerate (all 0s or all 1s).'''
    y, s = _to_np(y), _to_np(s)
    keys = ["auroc","auprc","hits@1","hits@3","hits@10","mrr","precision","recall","f1"]
    if len(y) == 0 or np.sum(y) == 0 or np.sum(y) == len(y):
        return {f"{prefix}{k}":0.0 for k in keys}
    tm = optimal_threshold_metrics(y, s)
    return {f"{prefix}auroc":float(roc_auc_score(y,s)), f"{prefix}auprc":float(average_precision_score(y,s)),
            f"{prefix}hits@1":hits_at_k(y,s,1), f"{prefix}hits@3":hits_at_k(y,s,3),
            f"{prefix}hits@10":hits_at_k(y,s,10), f"{prefix}mrr":mean_reciprocal_rank(y,s),
            f"{prefix}precision":tm["precision"], f"{prefix}recall":tm["recall"], f"{prefix}f1":tm["f1"]}
print("L5 ready")

### L6 · Leakage-safe train/val/test splits (the heart of an honest benchmark)

This is the most important methodological piece. If we naively split edges we leak the
answer, because the target edge would still be visible during message passing, and a
held-out disease might "see" its own indications. We prevent that:

- **`_build_base_graph`** removes **all** therapeutic drug↔disease edges from the
  message-passing graph and keeps only the *training* target edges. So the model never
  sees val/test labels in the graph it computes embeddings from.
- **Negatives** are sampled (random (drug, disease) pairs that are not known positives)
  so the model learns to discriminate, not just memorize.

Three evaluation **regimes**, increasingly hard:
1. **Transductive** — random edge split; all nodes seen during training.
2. **Inductive cold-disease (oncology)** — entire *diseases* held out; the model must
   generalize to cancers it never trained on. (The realistic repurposing setting.)
3. **Inductive cold-drug** — entire *drugs* held out.

Plus two **ablations** to probe what the GNN relies on:
- **`ablate_topology`** — `shuffle` (randomly rewire non-target edges) or `empty` (drop
  them) to test whether graph structure helps at all.
- **`drop_relations`** — remove specific edge families (e.g. drug→protein) to see which
  biology matters.

In [ ]:
from dataclasses import dataclass, field
from typing import Set, Tuple, List, Optional

# Normalized therapeutic relation names, for fast membership checks.
_THERAPEUTIC_NORM = {_norm_rel(r) for r in THERAPEUTIC_RELS}

@dataclass
class SplitData:
    '''Bundles a leakage-safe message-passing graph (`base`) with train/val/test labels.'''
    base: HeteroData; target_edge_type: tuple; regime: str
    train_label_index: torch.Tensor; train_label: torch.Tensor
    val_label_index: torch.Tensor; val_label: torch.Tensor
    test_label_index: torch.Tensor; test_label: torch.Tensor
    info: dict = field(default_factory=dict)

def _therapeutic_edge_types(data):
    '''All drug<->disease therapeutic edge types (both directions) — stripped to avoid leakage.'''
    out = []
    for et in data.edge_types:
        s, r, d = et
        if {s, d} == {DRUG_TYPE, DISEASE_TYPE} and r in _THERAPEUTIC_NORM: out.append(et)
    return out

def _sample_negatives(pos_set, src_pool, dst_pool, num, gen):
    '''Sample `num` (src, dst) pairs that are NOT in `pos_set` (negative training edges).'''
    if num <= 0 or src_pool.numel() == 0 or dst_pool.numel() == 0:
        return torch.empty((2,0), dtype=torch.long)
    neg_src, neg_dst, seen = [], [], set(); attempts = 0; max_attempts = max(2000, num*50)
    while len(neg_src) < num and attempts < max_attempts:
        batch = max(num*2, 128)                                  # over-sample, then filter
        si = src_pool[torch.randint(0, src_pool.numel(), (batch,), generator=gen)]
        di = dst_pool[torch.randint(0, dst_pool.numel(), (batch,), generator=gen)]
        for s, d in zip(si.tolist(), di.tolist()):
            key = (s, d)
            if key in pos_set or key in seen: continue          # skip true edges & duplicates
            seen.add(key); neg_src.append(s); neg_dst.append(d)
            if len(neg_src) >= num: break
        attempts += batch
    return torch.tensor([neg_src, neg_dst], dtype=torch.long)

def _build_base_graph(data, target_edge_type, train_target_cols):
    '''Copy the graph but: keep only TRAIN target edges, and delete ALL therapeutic edges.'''
    base = HeteroData()
    for nt in data.node_types:                                  # copy node stores verbatim
        for k, v in data[nt].items(): base[nt][k] = v
    therapeutic = set(_therapeutic_edge_types(data))
    for et in data.edge_types:
        if et == target_edge_type:                              # target: expose only training edges
            base[et].edge_index = data[et].edge_index[:, train_target_cols]
        elif et in therapeutic:                                 # other therapeutic edges: remove entirely
            base[et].edge_index = data[et].edge_index[:, :0]
        else:                                                   # all biology edges: keep
            base[et].edge_index = data[et].edge_index
    return base

def _make(base, target, regime, tr_pos, tr_neg, va_pos, va_neg, te_pos, te_neg, info):
    '''Assemble positives+negatives into labeled index/label tensors for each split.'''
    def cat(pos, neg):
        return (torch.cat([pos, neg], dim=1),
                torch.cat([torch.ones(pos.size(1)), torch.zeros(neg.size(1))]))
    tr_i, tr_l = cat(tr_pos, tr_neg); va_i, va_l = cat(va_pos, va_neg); te_i, te_l = cat(te_pos, te_neg)
    return SplitData(base, target, regime, tr_i, tr_l, va_i, va_l, te_i, te_l, info)

def transductive_split(data, target, val_frac=0.1, test_frac=0.2, neg_ratio=1.0, seed=0):
    '''Random edge split: shuffle target edges into train/val/test (all nodes seen).'''
    gen = torch.Generator().manual_seed(seed); s_t, _, d_t = target
    pos = data[target].edge_index; n = pos.size(1); perm = torch.randperm(n, generator=gen)
    n_test, n_val = int(round(test_frac*n)), int(round(val_frac*n))
    test_c, val_c, train_c = perm[:n_test], perm[n_test:n_test+n_val], perm[n_test+n_val:]
    tr_pos, va_pos, te_pos = pos[:, train_c], pos[:, val_c], pos[:, test_c]
    pos_set = {(int(s), int(d)) for s, d in zip(pos[0].tolist(), pos[1].tolist())}
    src_pool = torch.arange(int(data[s_t].num_nodes)); dst_pool = torch.arange(int(data[d_t].num_nodes))
    base = _build_base_graph(data, target, train_c)
    def neg(p): return _sample_negatives(pos_set, src_pool, dst_pool, int(round(p.size(1)*neg_ratio)), gen)
    info = {"regime":"transductive","train_pos":int(tr_pos.size(1)),
            "val_pos":int(va_pos.size(1)),"test_pos":int(te_pos.size(1))}
    return _make(base, target, "transductive", tr_pos, neg(tr_pos), va_pos, neg(va_pos), te_pos, neg(te_pos), info)

def inductive_node_split(data, target, holdout_side="dst", val_frac=0.1, test_frac=0.2,
                         neg_ratio=1.0, seed=0, restrict_oncology=False):
    '''Cold-node split: hold out whole drugs ('src') or whole diseases ('dst').

    Every edge touching a held-out node goes to val/test, so those nodes are never seen
    during training — the realistic 'generalize to a new disease/drug' setting.
    '''
    gen = torch.Generator().manual_seed(seed); s_t, _, d_t = target
    pos = data[target].edge_index
    side_row = 0 if holdout_side == "src" else 1
    side_type = s_t if holdout_side == "src" else d_t
    participating = torch.unique(pos[side_row])                  # nodes that have at least one target edge
    if restrict_oncology and holdout_side == "dst" and "is_oncology" in data[d_t]:
        onc = data[d_t].is_oncology                              # restrict held-out diseases to cancers
        participating = torch.tensor([n for n in participating.tolist() if bool(onc[n])], dtype=torch.long)
    perm = participating[torch.randperm(participating.numel(), generator=gen)]; n = participating.numel()
    n_test, n_val = max(1, int(round(test_frac*n))), max(1, int(round(val_frac*n)))
    test_nodes = set(perm[:n_test].tolist()); val_nodes = set(perm[n_test:n_test+n_val].tolist())
    held = test_nodes | val_nodes
    side_idx = pos[side_row].tolist(); tr_c, va_c, te_c = [], [], []
    for col, nd in enumerate(side_idx):                          # route each edge by its held-out endpoint
        (te_c if nd in test_nodes else va_c if nd in val_nodes else tr_c).append(col)
    tr_c = torch.tensor(tr_c, dtype=torch.long); tr_pos = pos[:, tr_c]
    va_pos = pos[:, torch.tensor(va_c, dtype=torch.long)] if va_c else pos[:, :0]
    te_pos = pos[:, torch.tensor(te_c, dtype=torch.long)] if te_c else pos[:, :0]
    pos_set = {(int(s), int(d)) for s, d in zip(pos[0].tolist(), pos[1].tolist())}
    other_type = d_t if holdout_side == "src" else s_t
    other_pool = torch.arange(int(data[other_type].num_nodes))
    train_side = torch.tensor(sorted(set(participating.tolist()) - held), dtype=torch.long)
    base = _build_base_graph(data, target, tr_c)
    def neg(p, side_pool):                                       # negatives must use the matching held-out nodes
        num = int(round(p.size(1)*neg_ratio))
        return (_sample_negatives(pos_set, side_pool, other_pool, num, gen) if holdout_side == "src"
                else _sample_negatives(pos_set, other_pool, side_pool, num, gen))
    def pool(s): return torch.tensor(sorted(s), dtype=torch.long)
    info = {"regime":f"inductive_cold_{holdout_side}","restrict_oncology":restrict_oncology,
            "n_test_nodes":len(test_nodes),"n_val_nodes":len(val_nodes),
            "train_pos":int(tr_pos.size(1)),"val_pos":int(va_pos.size(1)),"test_pos":int(te_pos.size(1))}
    return _make(base, target, info["regime"], tr_pos, neg(tr_pos, train_side),
                 va_pos, neg(va_pos, pool(val_nodes)), te_pos, neg(te_pos, pool(test_nodes)), info)

def make_split(data, target, regime, seed=0, holdout_side="dst", val_frac=0.1, test_frac=0.2,
               neg_ratio=1.0, restrict_oncology=False):
    '''Dispatch to the right split function by regime name.'''
    if regime == "transductive":
        return transductive_split(data, target, val_frac, test_frac, neg_ratio, seed)
    if regime in ("inductive","inductive_cold_dst","inductive_cold_src"):
        side = "src" if regime.endswith("src") else holdout_side
        return inductive_node_split(data, target, side, val_frac, test_frac, neg_ratio, seed, restrict_oncology)
    raise ValueError(regime)

def ablate_topology(split, mode, seed=0):
    '''Return a copy of the split with non-target edges 'shuffle'd (rewired) or 'empty'd (removed).'''
    gen = torch.Generator().manual_seed(seed); nb = HeteroData()
    for nt in split.base.node_types:
        for k, v in split.base[nt].items(): nb[nt][k] = v
    for et in split.base.edge_types:
        ei = split.base[et].edge_index
        if et == split.target_edge_type: nb[et].edge_index = ei; continue   # never touch target edges
        if mode == "empty":
            nb[et].edge_index = ei[:, :0]                                    # delete this relation
        elif mode == "shuffle":
            perm = torch.randperm(ei.size(1), generator=gen)                 # randomly rewire destinations
            nb[et].edge_index = torch.stack([ei[0], ei[1][perm]], 0)
        else:
            raise ValueError(mode)
    return SplitData(nb, split.target_edge_type, f"{split.regime}_ablate_{mode}", split.train_label_index,
                     split.train_label, split.val_label_index, split.val_label, split.test_label_index,
                     split.test_label, dict(split.info))

def drop_relations(split, drop_substrings):
    '''Return a copy of the split with edge families whose relation matches a substring removed.'''
    nb = HeteroData()
    for nt in split.base.node_types:
        for k, v in split.base[nt].items(): nb[nt][k] = v
    for et in split.base.edge_types:
        _, rel, _ = et
        if et != split.target_edge_type and any(sub in rel for sub in drop_substrings):
            continue                                                          # skip = drop this relation
        nb[et].edge_index = split.base[et].edge_index
    return SplitData(nb, split.target_edge_type, f"{split.regime}_drop", split.train_label_index,
                     split.train_label, split.val_label_index, split.val_label, split.test_label_index,
                     split.test_label, dict(split.info))
print("L6 ready")

### L7 · Training loops

One training loop per model family. All of them:
- optimize **binary cross-entropy** on positive vs negative edges,
- track **validation AUROC** every epoch and keep the **best** weights (early stopping),
- so we never report an over-fit final epoch.

`train_gnn_joint` is special: it trains the GNN on the link objective **plus** a
**contrastive mechanism objective** — for each curated (drug, disease) pair it must rank
the true bridge **gene** above several *degree-matched decoy* genes (so it can't cheat by
just picking popular genes). This is what gives the GNN its mechanism-recovery ability.

`set_all_seeds` pins every RNG for reproducibility; `evaluate_model` reports test (or val)
metrics for any model type.

In [ ]:
import copy, random

def set_all_seeds(seed):
    '''Pin Python/NumPy/Torch RNGs so a run is reproducible.'''
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

@torch.no_grad()
def _eval_encoder(model, base, et, eli, labels, device):
    '''Encode the graph once, score the labeled edges, and compute metrics.'''
    model.eval(); z = model.encode(base); scores = torch.sigmoid(model.decode(z, et, eli)).cpu()
    return compute_all_metrics(labels.cpu(), scores)

def train_gnn(model, split, device, epochs=50, patience=10, lr=5e-3, weight_decay=1e-5):
    '''Train the HeteroGNN on link prediction with early stopping on val AUROC.'''
    model = model.to(device); et = split.target_edge_type
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    y = split.train_label.float().to(device); best, best_state, wait = -1.0, None, 0
    for _ in range(epochs):
        model.train(); opt.zero_grad()
        z = model.encode(split.base)                                       # message passing
        loss = F.binary_cross_entropy_with_logits(model.decode(z, et, split.train_label_index), y)
        loss.backward(); opt.step()
        val = _eval_encoder(model, split.base, et, split.val_label_index, split.val_label, device)["auroc"]
        if val > best:                                                     # remember the best checkpoint
            best, wait = val, 0
            best_state = copy.deepcopy({k: v.cpu() for k, v in model.state_dict().items()})
        else:
            wait += 1
            if wait >= patience: break                                     # early stop
    if best_state is not None: model.load_state_dict(best_state)
    return model

def train_mlp(model, split, device, epochs=200, patience=20, lr=5e-3, weight_decay=1e-5):
    '''Train the features-only MLP baseline (same loop, no graph messages).'''
    model = model.to(device); et = split.target_edge_type
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    y = split.train_label.float().to(device); best, best_state, wait = -1.0, None, 0
    for _ in range(epochs):
        model.train(); opt.zero_grad(); z = model.encode(split.base)
        loss = F.binary_cross_entropy_with_logits(model.decode(z, et, split.train_label_index), y)
        loss.backward(); opt.step()
        val = _eval_encoder(model, split.base, et, split.val_label_index, split.val_label, device)["auroc"]
        if val > best: best, wait = val, 0; best_state = copy.deepcopy({k: v.cpu() for k, v in model.state_dict().items()})
        else:
            wait += 1
            if wait >= patience: break
    if best_state is not None: model.load_state_dict(best_state)
    return model

@torch.no_grad()
def _eval_kge(model, eli, labels):
    model.eval(); return compute_all_metrics(labels.cpu(), torch.sigmoid(model.score(eli)).cpu())

def train_kge(model, split, device, epochs=300, patience=30, lr=1e-2, weight_decay=1e-6):
    '''Train the DistMult KGE baseline (scores from learned embeddings only).'''
    model = model.to(device); opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    y = split.train_label.float().to(device); best, best_state, wait = -1.0, None, 0
    for _ in range(epochs):
        model.train(); opt.zero_grad()
        loss = F.binary_cross_entropy_with_logits(model.score(split.train_label_index), y)
        loss.backward(); opt.step()
        val = _eval_kge(model, split.val_label_index, split.val_label)["auroc"]
        if val > best: best, wait = val, 0; best_state = copy.deepcopy({k: v.cpu() for k, v in model.state_dict().items()})
        else:
            wait += 1
            if wait >= patience: break
    if best_state is not None: model.load_state_dict(best_state)
    return model

def train_gnn_joint(model, split, device, mech, decoys, lam=1.0, n_neg=8, mech_batch=256,
                    epochs=60, lr=5e-3, weight_decay=1e-5):
    '''Joint training: link-prediction loss + contrastive mechanism (bridge-gene) loss.

    For each curated (drug, disease, bridge_gene) example we build a candidate set of
    [true gene + n_neg degree-matched decoys] and use cross-entropy to push the true
    gene's mechanism-score to the top. `lam` weights this objective against the link loss.
    '''
    model = model.to(device); et = split.target_edge_type
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    y = split.train_label.float().to(device)
    md_, mc, mg = mech.drug, mech.dis, mech.gene; M = int(md_.numel())
    for _ in range(epochs):
        model.train(); opt.zero_grad(); z = model.encode(split.base)
        link = F.binary_cross_entropy_with_logits(model.decode(z, et, split.train_label_index), y)
        mech_loss = torch.tensor(0.0, device=device)
        if M > 0 and lam > 0:
            b = min(mech_batch, M); idx = torch.randint(0, M, (b,))         # sample a batch of examples
            bd, bc, bg = md_[idx], mc[idx], mg[idx]
            # For each example draw n_neg decoy genes from the same degree bucket as the true gene.
            neg = torch.tensor([decoys.sample(int(bg[j]), {int(bg[j])}, n_neg) for j in range(b)], dtype=torch.long)
            cand = torch.cat([bg.view(-1,1), neg], dim=1); K = cand.size(1) # column 0 = the true gene
            fd = bd.view(-1,1).expand(-1,K).reshape(-1); fc = bc.view(-1,1).expand(-1,K).reshape(-1)
            fg = cand.reshape(-1)
            logits = model.score_mechanism(z, fd, fg, fc).view(b, K)
            # Cross-entropy with target 0 => "rank the true gene (col 0) above the decoys".
            mech_loss = F.cross_entropy(logits, torch.zeros(b, dtype=torch.long, device=logits.device))
        (link + lam*mech_loss).backward(); opt.step()
    return model

def evaluate_model(model, split, device, which="test"):
    '''Return metrics on the chosen split ('test' or 'val') for any model type.'''
    eli = getattr(split, f"{which}_label_index"); lab = getattr(split, f"{which}_label")
    if isinstance(model, DistMultKGE): return _eval_kge(model, eli, lab)
    return _eval_encoder(model, split.base, split.target_edge_type, eli, lab, device)
print("L7 ready")

### L8 · Tuned XGBoost tabular baseline

The toughest competitor. We turn each (drug, disease) pair into a flat feature vector by
**concatenating the two nodes' feature embeddings**, then train gradient-boosted trees on
exactly the same train/val/test split as the GNN. In full mode we tune hyper-parameters
with Optuna (validation AUROC objective) and use early stopping.

This baseline matters because — spoiler — it **matches or beats the GNN on link ranking**,
which is the honest finding the project foregrounds: the GNN's value is *not* raw ranking,
it's the *mechanism* it can additionally produce.

In [ ]:
def _pair_features(data, et, eli):
    '''Concatenate source- and destination-node features for each labeled edge -> tabular X.'''
    s_t, _, d_t = et
    return np.concatenate([data[s_t].x[eli[0]].cpu().numpy(),
                           data[d_t].x[eli[1]].cpu().numpy()], axis=1)

def run_xgboost(split, data, seed=0, n_estimators=400, max_depth=6, lr=0.1, tune=False, n_trials=20):
    '''Train (optionally Optuna-tuned) XGBoost on pair features; return test metrics.'''
    import xgboost as xgb
    et = split.target_edge_type
    Xtr = _pair_features(data, et, split.train_label_index); ytr = split.train_label.cpu().numpy()
    Xva = _pair_features(data, et, split.val_label_index);   yva = split.val_label.cpu().numpy()
    Xte = _pair_features(data, et, split.test_label_index);  yte = split.test_label.cpu().numpy()
    base = dict(n_estimators=600, max_depth=max_depth, learning_rate=lr, subsample=0.9,
                colsample_bytree=0.9, eval_metric="logloss", tree_method="hist",
                random_state=seed, n_jobs=-1, early_stopping_rounds=30)
    if tune:                                                 # search a few configs, keep best val AUROC
        import optuna
        def objective(trial):
            p = dict(n_estimators=600, max_depth=trial.suggest_int("max_depth",3,9),
                     learning_rate=trial.suggest_float("learning_rate",0.03,0.3,log=True),
                     subsample=trial.suggest_float("subsample",0.7,1.0),
                     colsample_bytree=trial.suggest_float("colsample_bytree",0.7,1.0),
                     min_child_weight=trial.suggest_int("min_child_weight",1,8),
                     eval_metric="logloss", tree_method="hist", random_state=seed,
                     n_jobs=-1, early_stopping_rounds=30)
            m = xgb.XGBClassifier(**p); m.fit(Xtr, ytr, eval_set=[(Xva, yva)], verbose=False)
            return roc_auc_score(yva, m.predict_proba(Xva)[:,1])
        optuna.logging.set_verbosity(optuna.logging.WARNING)
        study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=seed))
        study.optimize(objective, n_trials=n_trials, show_progress_bar=False); base.update(study.best_params)
    model = xgb.XGBClassifier(**base); model.fit(Xtr, ytr, eval_set=[(Xva, yva)], verbose=False)
    return compute_all_metrics(yte, model.predict_proba(Xte)[:,1])
print("L8 ready")

---
# Workflow

## 2. PrimeKG: download, build, load, inspect

We download PrimeKG (once), build/load the heterogeneous graph, attach node features
(hashing in fast mode, SentenceTransformer in full mode), and print a summary plus a
per-node-type table. After this cell, `data` is the graph and `target` is the
`(drug, indication, disease)` edge type we predict.

In [ ]:
download_primekg()                               # one-time ~980 MB download (skipped if cached)
if not HETERODATA_PT.exists():
    build_hetero_from_primekg()                  # parse CSV -> HeteroData (cached)
data, targets = load_primekg(with_features=True, force_fallback_features=USE_FALLBACK_FEATS)
target = targets["indication"]                   # our main prediction target edge type
print(graph_summary(data, targets))

In [ ]:
# A quick per-node-type table so you can see the scale and feature dimension of each type.
import pandas as pd
rows = [{"node_type":nt,
         "num_nodes":int(data[nt].num_nodes),
         "feature_dim":int(data[nt].x.size(1)) if "x" in data[nt] else None,
         "example_names":", ".join(str(n) for n in list(getattr(data[nt],"node_names",[]))[:2])}
        for nt in data.node_types]
display(pd.DataFrame(rows).sort_values("num_nodes", ascending=False).reset_index(drop=True))
print("oncology diseases:", int(data[DISEASE_TYPE].is_oncology.sum()),
      "| indication edges:", int(data[target].edge_index.size(1)))

## 3. Aim 1 — Candidate generation: link benchmark + ablations

We train all four models across the three regimes, average over `SEEDS`, run paired
t-tests (GNN vs each baseline), then ablate the GNN's graph topology and relation
families. **Watch for the honest finding:** tuned XGBoost on the same features ties or
beats the GNN on link ranking — so ranking alone doesn't justify the graph.

In [ ]:
# Map each node type to its feature dimension (models need this to size their input layers).
in_dims = {nt: int(data[nt].x.size(1)) for nt in data.node_types}
REGIMES = ["transductive", "inductive_cold_dst", "inductive_cold_src"]
REGIME_LABEL = {"transductive":"Transductive",
                "inductive_cold_dst":"Inductive cold-disease (onc)",
                "inductive_cold_src":"Inductive cold-drug"}
MODELS = ["GNN","XGBoost","MLP","KGE"]

def train_eval_one(name, split, seed):
    '''Train one model on a split and return its test metrics dict.'''
    set_all_seeds(seed)
    if name == "GNN":
        m = HeteroGNN(list(data.node_types), list(split.base.edge_types), in_dims,
                      hidden=128, num_layers=2, dropout=0.3)
        return evaluate_model(train_gnn(m, split, DEVICE, epochs=GNN_EPOCHS, patience=PATIENCE), split, DEVICE)
    if name == "MLP":
        m = FeatureMLP(list(data.node_types), in_dims, hidden=128, dropout=0.3)
        return evaluate_model(train_mlp(m, split, DEVICE, epochs=MLP_EPOCHS, patience=PATIENCE), split, DEVICE)
    if name == "KGE":
        m = DistMultKGE(DRUG_TYPE, DISEASE_TYPE, int(data[DRUG_TYPE].num_nodes),
                        int(data[DISEASE_TYPE].num_nodes), dim=128)
        return evaluate_model(train_kge(m, split, DEVICE, epochs=KGE_EPOCHS, patience=PATIENCE), split, DEVICE)
    if name == "XGBoost":
        return run_xgboost(split, data, seed=seed, n_estimators=XGB_ESTIMATORS, tune=XGB_TUNE, n_trials=XGB_TRIALS)

# Train every (regime, model, seed) combination and stash the metrics.
per_seed = {r: {m: {} for m in MODELS} for r in REGIMES}
for regime in REGIMES:
    print(f"\n=== REGIME: {regime} ===")
    for seed in SEEDS:
        kw = {"restrict_oncology": True} if regime == "inductive_cold_dst" else {}  # hold out cancers only
        split = make_split(data, target, regime, seed=seed, **kw)
        for name in MODELS:
            mt = train_eval_one(name, split, seed); per_seed[regime][name][seed] = mt
            print(f"  seed {seed} {name:8s} auroc={mt['auroc']:.4f} auprc={mt['auprc']:.4f}")

In [ ]:
# Aggregate AUROC over seeds into a regime x model table.
import numpy as np
def agg(vals): a = np.asarray(vals, float); return a.mean(), a.std()
print("Test AUROC (mean over seeds):")
table = {}
for regime in REGIMES:
    table[REGIME_LABEL[regime]] = {}
    for name in MODELS:
        vals = [per_seed[regime][name][s]["auroc"] for s in SEEDS]; m, sd = agg(vals)
        table[REGIME_LABEL[regime]][name] = m
import pandas as pd
df_grid = pd.DataFrame(table).T[MODELS].round(3); display(df_grid)

# Paired t-tests: is the GNN significantly better than each baseline? (needs >1 seed)
from scipy import stats as _stats
if len(SEEDS) > 1:
    print("\nPaired t-tests (one-sided, GNN > baseline, AUROC):")
    for regime in REGIMES:
        gv = [per_seed[regime]["GNN"][s]["auroc"] for s in SEEDS]
        for b in ["XGBoost","MLP","KGE"]:
            bv = [per_seed[regime][b][s]["auroc"] for s in SEEDS]
            t, p = _stats.ttest_rel(gv, bv); p1 = p/2 if t > 0 else 1 - p/2
            print(f"  [{regime}] GNN vs {b}: mean_diff={np.mean(gv)-np.mean(bv):+.3f} p={p1:.4f}")
else:
    print("\n(Single seed in FAST_MODE: set FAST_MODE=False for multi-seed significance tests.)")

# Bar chart of the AUROC grid.
import matplotlib.pyplot as plt
ax = df_grid.plot(kind="bar", figsize=(10,5), rot=10); ax.axhline(0.5, color="gray", ls="--", lw=1)
ax.set_ylabel("Test AUROC"); ax.set_ylim(0.4, 1.0); ax.set_title("GNN vs baselines (PrimeKG link prediction)")
plt.tight_layout(); plt.savefig(FIGURES_DIR/"main_results.png", dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
# --- Topology ablation (cold-disease regime, where graph structure should matter most) ---
# intact = real graph; shuffle = randomly rewired non-target edges; empty = no non-target edges.
topo = {}
for mode in ["intact","shuffle","empty"]:
    vals = []
    for seed in ABLATION_SEEDS:
        sp = make_split(data, target, "inductive_cold_dst", seed=seed, restrict_oncology=True)
        s = sp if mode == "intact" else ablate_topology(sp, mode, seed=seed)
        set_all_seeds(seed)
        m = HeteroGNN(list(data.node_types), list(s.base.edge_types), in_dims, hidden=128, num_layers=2, dropout=0.3)
        vals.append(evaluate_model(train_gnn(m, s, DEVICE, epochs=GNN_EPOCHS, patience=PATIENCE), s, DEVICE)["auroc"])
    topo[mode] = float(np.mean(vals)); print(f"  topology[{mode:7s}] GNN AUROC={topo[mode]:.4f}")

# --- Relation ablation: drop specific biology edge families and see the impact ---
rel_groups = {"drop_drug_protein":["drug_protein","carrier","enzyme","target","transporter"],
              "drop_disease_protein":["disease_protein"],
              "drop_pathway":["pathway"]}
rel = {}
for nm, subs in rel_groups.items():
    vals = []
    for seed in ABLATION_SEEDS:
        sp = make_split(data, target, "inductive_cold_dst", seed=seed, restrict_oncology=True)
        s = drop_relations(sp, subs); set_all_seeds(seed)
        m = HeteroGNN(list(data.node_types), list(s.base.edge_types), in_dims, hidden=128, num_layers=2, dropout=0.3)
        vals.append(evaluate_model(train_gnn(m, s, DEVICE, epochs=GNN_EPOCHS, patience=PATIENCE), s, DEVICE)["auroc"])
    rel[nm] = float(np.mean(vals)); print(f"  relation[{nm}] GNN AUROC={rel[nm]:.4f}")

plt.figure(figsize=(5.5,4)); plt.bar(list(topo), list(topo.values()), color=["#4C72B0","#DD8452","#C44E52"])
plt.axhline(0.5, color="gray", ls="--", lw=1); plt.ylabel("GNN AUROC (cold-disease)"); plt.ylim(0.4,1.0)
plt.title("Topology ablation"); plt.tight_layout()
plt.savefig(FIGURES_DIR/"ablation_topology.png", dpi=150, bbox_inches="tight"); plt.show()

## Recap & next: Part 2

You built the knowledge graph, trained four models under three regimes, and saw the key result: **a tuned XGBoost matches/beats the GNN on link ranking**, and the topology ablation shows the GNN leans heavily on node features, not deep graph structure, for *ranking*.

So why use a graph at all? Because a ranking score can't tell you **why** a drug might work. **Continue to `oncoevidence_part2_mechanisms.ipynb`**, where we extract the multi-hop biological *mechanism* behind each candidate — something no tabular model can produce.